# BCB Non-clone Dataset Analysis

Reads the prepared non-clone artifacts used as the shared negative pool for BCB Type 1-4 tuning. This notebook does not rescan the full BCB dump.

In [1]:
from pathlib import Path
import html
import json
import pickle
import random
import sys

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import HTML, display
from tqdm.auto import tqdm


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "spectral_code").exists() and (candidate / "pipelines").exists():
            return candidate
    raise RuntimeError("Project root not found.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from spectral_code.utils.dataset_paths import bcb_type_dir, output_root_for

VARIANT = "non_clone"
DATA_DIR = bcb_type_dir(VARIANT)
OUTPUT_ROOT = output_root_for("bcb", VARIANT)
GRAPH_MANIFEST_PATH = OUTPUT_ROOT / "clean_graphs" / "graph_shards_manifest.json"
SPECTRAL_MANIFEST_PATH = OUTPUT_ROOT / "spectral_features" / "spectral_features_manifest.json"
PIPELINE_TIMINGS_PATH = OUTPUT_ROOT / "pipeline_timings.json"
TIMING_STATS_PATH = OUTPUT_ROOT / "timing_stats.json"
GRAPH_TYPES = ["ast", "cfg", "ddg", "pdg", "cpg"]
BASE_GRAPH_TYPES = ["ast", "cfg", "ddg", "pdg"]

print("Project root:", PROJECT_ROOT)
print("Prepared non-clone data:", DATA_DIR)
print("Output root:", OUTPUT_ROOT)
print("Graph manifest:", GRAPH_MANIFEST_PATH if GRAPH_MANIFEST_PATH.exists() else "not found yet")
print("Spectral manifest:", SPECTRAL_MANIFEST_PATH if SPECTRAL_MANIFEST_PATH.exists() else "not found yet")

METADATA_PATH = DATA_DIR / "metadata.json"
metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8")) if METADATA_PATH.exists() else {}
pipeline_timings = json.loads(PIPELINE_TIMINGS_PATH.read_text(encoding="utf-8")) if PIPELINE_TIMINGS_PATH.exists() else {"stages": {}}
timing_stats = json.loads(TIMING_STATS_PATH.read_text(encoding="utf-8")) if TIMING_STATS_PATH.exists() else {}


Project root: c:\Users\koush\PyProjects\spectrals\Spectral-Software
Prepared non-clone data: C:\Users\koush\PyProjects\spectrals\data\bcb\non_clone
Output root: C:\Users\koush\PyProjects\spectrals\outputs\bcb\non_clone
Graph manifest: C:\Users\koush\PyProjects\spectrals\outputs\bcb\non_clone\clean_graphs\graph_shards_manifest.json
Spectral manifest: C:\Users\koush\PyProjects\spectrals\outputs\bcb\non_clone\spectral_features\spectral_features_manifest.json


c:\Users\koush\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def count_lines(path: Path) -> int:
    if not path.exists():
        return 0
    with path.open("rb") as f:
        return sum(1 for _ in f)


def label_distribution(path: Path) -> dict[int, int]:
    counts: dict[int, int] = {}
    if not path.exists():
        return counts
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 3:
                continue
            label = int(parts[2])
            counts[label] = counts.get(label, 0) + 1
    return counts


def unique_code_ids_in_pairs(path: Path) -> int:
    ids = set()
    if not path.exists():
        return 0
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 2:
                ids.add(parts[0])
                ids.add(parts[1])
    return len(ids)


summary = {
    "prepared_data_dir": str(DATA_DIR),
    "code_rows": count_lines(DATA_DIR / "data.jsonl"),
    "train_pairs": count_lines(DATA_DIR / "train.txt"),
    "train_positive_pairs": count_lines(DATA_DIR / "train_positives.txt"),
    "type_labels": count_lines(DATA_DIR / "type_labels.tsv"),
    "unique_code_ids_in_pairs": unique_code_ids_in_pairs(DATA_DIR / "train.txt"),
    "label_distribution": label_distribution(DATA_DIR / "train.txt"),
    "metadata_total_pairs": metadata.get("total_pairs"),
    "metadata_written_functions": metadata.get("written_functions"),
    "metadata_non_clones": metadata.get("non_clones"),
    "graph_manifest_exists": GRAPH_MANIFEST_PATH.exists(),
    "spectral_manifest_exists": SPECTRAL_MANIFEST_PATH.exists(),
}

display(pd.DataFrame([summary]).T.rename(columns={0: "value"}))


,value
prepared_data_dir,C:\Users\koush\PyProjects\spectrals\data\bcb\n...
code_rows,55693
train_pairs,260708
train_positive_pairs,0
type_labels,0
unique_code_ids_in_pairs,55677
label_distribution,{0: 260708}
metadata_total_pairs,260785
metadata_written_functions,55693
metadata_non_clones,260785


In [3]:
graph_stats = {
    "dot_mapped_ast": timing_stats.get("dot_mapped_ast"),
    "dot_missing_ast": timing_stats.get("dot_missing_ast"),
    "dot_mapped_cfg": timing_stats.get("dot_mapped_cfg"),
    "dot_missing_cfg": timing_stats.get("dot_missing_cfg"),
    "dot_mapped_ddg": timing_stats.get("dot_mapped_ddg"),
    "dot_missing_ddg": timing_stats.get("dot_missing_ddg"),
    "dot_mapped_pdg": timing_stats.get("dot_mapped_pdg"),
    "dot_missing_pdg": timing_stats.get("dot_missing_pdg"),
    "skipped_graph_layers_pipeline01": timing_stats.get("skipped_graph_layers_pipeline01"),
    "total_methods_cleaned": timing_stats.get("total_methods_cleaned"),
    "total_layers_cleaned": timing_stats.get("total_layers_cleaned"),
    "clean_graphs_manifest": timing_stats.get("clean_graphs_manifest"),
}

display(pd.DataFrame([graph_stats]).T.rename(columns={0: "value"}))


,value
dot_mapped_ast,55678
dot_missing_ast,0
dot_mapped_cfg,55678
dot_missing_cfg,0
dot_mapped_ddg,55678
dot_missing_ddg,0
dot_mapped_pdg,55678
dot_missing_pdg,0
skipped_graph_layers_pipeline01,8
total_methods_cleaned,55678


In [4]:
def reservoir_sample_pairs(path: Path, n: int, seed: int) -> list[tuple[str, str, int]]:
    rng = random.Random(seed)
    sample: list[tuple[str, str, int]] = []
    seen = 0
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            left, right, raw_label = line.rstrip("\n").split("\t")
            row = (left, right, int(raw_label))
            seen += 1
            if len(sample) < n:
                sample.append(row)
            else:
                replace_at = rng.randrange(seen)
                if replace_at < n:
                    sample[replace_at] = row
    return sample


def load_code_for_ids(path: Path, wanted_ids: set[str]) -> dict[str, str]:
    code_map = {}
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            function_id = str(row["idx"])
            if function_id in wanted_ids:
                code_map[function_id] = row.get("func", "")
                if len(code_map) == len(wanted_ids):
                    break
    return code_map


example_pairs = reservoir_sample_pairs(DATA_DIR / "train.txt", n=3, seed=42)
wanted_ids = {item for pair in example_pairs for item in pair[:2]}
code_map = load_code_for_ids(DATA_DIR / "data.jsonl", wanted_ids)
example_pairs = [pair for pair in example_pairs if pair[0] in code_map and pair[1] in code_map]

print("Loaded code snippets:", len(code_map))
print("Non-clone examples:", len(example_pairs))


Loaded code snippets: 5
Non-clone examples: 3


In [5]:
def render_code_pair(pair: tuple[str, str, int], title: str) -> HTML:
    left_id, right_id, label = pair
    left_code = html.escape(code_map.get(left_id, ""))
    right_code = html.escape(code_map.get(right_id, ""))
    return HTML(f'''
    <div style="margin: 18px 0 28px 0;">
      <h3 style="margin: 0 0 8px 0; font-family: system-ui, sans-serif;">{html.escape(title)} (label={label})</h3>
      <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 12px; align-items: start;">
        <div style="border: 1px solid #d0d7de; border-radius: 6px; overflow: hidden;">
          <div style="padding: 6px 10px; background: #f6f8fa; font-family: system-ui, sans-serif; font-size: 13px;">left id: {html.escape(left_id)}</div>
          <pre style="margin: 0; padding: 12px; overflow-x: auto; white-space: pre-wrap; font-size: 12px; line-height: 1.35;">{left_code}</pre>
        </div>
        <div style="border: 1px solid #d0d7de; border-radius: 6px; overflow: hidden;">
          <div style="padding: 6px 10px; background: #f6f8fa; font-family: system-ui, sans-serif; font-size: 13px;">right id: {html.escape(right_id)}</div>
          <pre style="margin: 0; padding: 12px; overflow-x: auto; white-space: pre-wrap; font-size: 12px; line-height: 1.35;">{right_code}</pre>
        </div>
      </div>
    </div>
    ''')


for idx, pair in enumerate(example_pairs, start=1):
    display(render_code_pair(pair, f"Non-clone example {idx}"))


In [6]:
def stage_runtime_row(stage: str) -> dict:
    record = pipeline_timings.get("stages", {}).get(stage, {})
    seconds = record.get("seconds")
    minutes = record.get("minutes")
    if minutes is None and isinstance(seconds, (int, float)):
        minutes = seconds / 60
    return {
        "stage": stage,
        "seconds": round(seconds, 2) if isinstance(seconds, (int, float)) else None,
        "minutes": round(minutes, 2) if isinstance(minutes, (int, float)) else None,
        "updated_at_utc": record.get("updated_at_utc"),
        "status": "recorded" if isinstance(seconds, (int, float)) else "not recorded yet",
        "source": str(PIPELINE_TIMINGS_PATH),
    }


runtime_df = pd.DataFrame([
    stage_runtime_row("01_extract_data"),
    stage_runtime_row("02_extract_raw_graphs"),
    stage_runtime_row("02_build_graph_db"),
    stage_runtime_row("02_extract_graphs"),
    stage_runtime_row("03_extract_spectral_features"),
])
display(runtime_df)


,stage,seconds,minutes,updated_at_utc,status,source
0,01_extract_data,88.68,1.48,2026-06-30T06:06:10+00:00,recorded,C:\Users\koush\PyProjects\spectrals\outputs\bc...
1,02_extract_raw_graphs,6790.88,113.18,2026-06-30T08:21:25+00:00,recorded,C:\Users\koush\PyProjects\spectrals\outputs\bc...
2,02_build_graph_db,1182.44,19.71,2026-06-30T08:21:25+00:00,recorded,C:\Users\koush\PyProjects\spectrals\outputs\bc...
3,02_extract_graphs,8112.07,135.20,2026-06-30T08:21:25+00:00,recorded,C:\Users\koush\PyProjects\spectrals\outputs\bc...
4,03_extract_spectral_features,3971.00,66.18,2026-06-30T09:27:48+00:00,recorded,C:\Users\koush\PyProjects\spectrals\outputs\bc...
